In [1]:
# main.py
import pulp
from src.domain import SupplyChainNetwork, ProductType, NodeType
from src.optimizer import SupplyChainSolver
from src.loaders import preparation
import pandas as pd

pd.set_option('display.float_format', '{:,.0f}'.format)

In [2]:
# ========================================================
# 1. FINANCIAL AND TECHNICAL PARAMETERS (SCALED)
# ========================================================
YIELD_CAKE = 0.78
YIELD_OIL = 0.19

# Scale down all costs. Instead of millions, use tens and thousands.
# RULE: Penalties MUST be strictly higher than Rewards to prevent infinite loops!

P_CONTRACT = 50_000.0        # High penalty for missing a real contract
P_MAGIC_GEN = 100_000.0      # Massive penalty for breaking physics (must be highest!)
P_DUMMY = 80_000.0           # Penalty for buying/selling out of the system

# Rewards should be small nudges just to prioritize paths, not get-rich-quick schemes
EXPORT_REWARD = -1_000.0     
DOMESTIC_REWARD = -500.0

In [3]:
network = SupplyChainNetwork()

solver = SupplyChainSolver(name="Soy_Optimization_Final")

In [4]:
network = preparation(network, paths={
    'production': 'data/production.csv',
    'silos': 'data/silos.csv',
    'ports': 'data/branches.csv',
    'train': 'data/train_flows.csv',
    'fixed_flows': 'data/branches.csv',
    'rail_flows': 'data/train_flows.csv',
    'truck_costs': 'data/cost.csv',
    'industrial_capacity': 'data/industrial_capacity.csv'
})

Starting Data Pipeline Preparation...
[ProductionNodes] Loading data from data/production.csv...
[SiloNodes] Loading data from data/silos.csv...
[PortNodes] Loading data from data/branches.csv...
[TrainNodes] Loading data from data/train_flows.csv...
[ProcessingNodes] Loading data from data/industrial_capacity.csv...
[FixedFlowsConstraints] Loading data from data/branches.csv...
⚠️ Created Ghost Industry: PROCESSING_BR_3303906 (Cap: 36753t)
⚠️ Created Ghost Industry: PROCESSING_BR_4300059 (Cap: 2885t)
[TrainConstraints] Loading data from data/train_flows.csv...
[TruckCostMatrix] Loading data from data/cost.csv...
[TruckCostMatrix] Created 10975555 valid edges. Ignored 17468685.
Preparation Complete. Total Nodes: 5371 | Total Edges: 10975831


# Add flow variables

In [5]:

solver.add_flow_variables(network.edges, [p for p in ProductType])


# Create optional Slacks

In [6]:
industry_ids = [n.id for n in network.nodes.values() if n.type == NodeType.PROCESSING]
solver.add_supply_slacks(node_ids=industry_ids, penalty_cost=P_DUMMY, product=ProductType.SOYBEANS)

surplus_nodes = [n.id for n in network.nodes.values() if n.type in [NodeType.PORT, NodeType.SILO_AGGREGATOR, NodeType.PRODUCTION]]
solver.add_sink_slacks(node_ids=surplus_nodes, penalty_cost=P_DUMMY)

# Create indexed to access nodes and edges

In [7]:
solver.prepare_flow_indexes()

Preparing high-performance flow indexes...


# Add transportation costs

In [ ]:
# Transport Costs
transport_costs = []
for edge in network.edges:
    for prod in ProductType:
        var = solver.flow_vars.get((edge.source_id, edge.target_id, edge.mode, prod))
        if var: transport_costs.append(var * edge.unit_cost)

solver.add_to_objective(pulp.lpSum(transport_costs))

# Production Constraints

In [9]:
domestic_vars = {}
waste_vars = {}
magic_gen_vars = {}
export_vars = {}

for node_id, node in network.nodes.items():
    if node.type == NodeType.PRODUCTION:
        beans_out = pulp.lpSum(solver.get_outbound_flows(node_id, ProductType.SOYBEANS))
        cake_out = pulp.lpSum(solver.get_outbound_flows(node_id, ProductType.SOYBEAN_CAKE))
        oil_out = pulp.lpSum(solver.get_outbound_flows(node_id, ProductType.SOYBEAN_OIL))
        
        solver.add_custom_constraint(beans_out == node.production, f"Supply_{node_id}")
        solver.add_custom_constraint((cake_out + oil_out) == 0, f"No_Magic_Gen_{node_id}")

# Silo Constraints

In [ ]:
for node_id, node in network.nodes.items():    
    if node.type in [NodeType.HUB, NodeType.SILO_AGGREGATOR, NodeType.SILO_LOCAL]:       
        for prod in ProductType:
            flow_in = pulp.lpSum(solver.get_inbound_flows(node_id, prod))
            flow_out = pulp.lpSum(solver.get_outbound_flows(node_id, prod))
            slack_out = solver.get_slack_var(node_id, 'sink', prod) # The default '0' helps if it doesn't exist

            solver.add_custom_constraint((flow_in) == (flow_out + slack_out), f"Mass_Bal_{node_id}_{prod.value}")

            if node.type == NodeType.HUB:
                solver.add_custom_constraint(flow_in <= (node.capacity * 20.0), f"Turnover_Hub_{node_id}_{prod.value}")
            elif node.type == NodeType.SILO_AGGREGATOR:
                solver.add_custom_constraint(flow_in <= (node.capacity * 10.0), f"Turnover_Agg_{node_id}_{prod.value}")
            elif node.type == NodeType.SILO_LOCAL:
                solver.add_custom_constraint(flow_in <= (node.capacity * 1.2), f"Turnover_Loc_{node_id}_{prod.value}")

# Train Constraint

In [11]:
for node_id, node in network.nodes.items():
    if node.type == NodeType.TRAIN:
        for prod in ProductType:
            flow_in = pulp.lpSum(solver.get_inbound_flows(node_id, prod))
            flow_out = pulp.lpSum(solver.get_outbound_flows(node_id, prod))
            solver.add_custom_constraint(flow_in == flow_out, f"Train_Bal_{node_id}_{prod.value}")


# Industry Constraint

In [12]:

for node_id, node in network.nodes.items():

    if node.type == NodeType.PROCESSING:

        beans_in = pulp.lpSum(solver.get_inbound_flows(node_id, ProductType.SOYBEANS))
        beans_out = pulp.lpSum(solver.get_outbound_flows(node_id, ProductType.SOYBEANS))

        slack_in_beans = solver.get_slack_var(node_id, 'supply', ProductType.SOYBEANS)
        
        # The Crushing Carousel
        crushed_beans = beans_in + slack_in_beans  
        solver.add_custom_constraint(crushed_beans >= 0, f"Positive_Crush_{node_id}")
        solver.add_custom_constraint(beans_out == 0, f"No_Beans_Out_{node_id}")

        for prod in [ProductType.SOYBEAN_CAKE, ProductType.SOYBEAN_OIL]:
            prod_out = pulp.lpSum(solver.get_outbound_flows(node_id, prod))
            yield_rate = YIELD_CAKE if prod == ProductType.SOYBEAN_CAKE else YIELD_OIL

            # Yield (Supply = Demand)
            solver.add_custom_constraint(
                (crushed_beans * yield_rate) >= (prod_out),
                f"Yield_{prod.value}_{node_id}"
            )

            # Quota Rule (Processing Contract)
            required_amount = node.contract_demands.get(prod, 0.0)
            if required_amount > 0.001:
                slack_quota = pulp.LpVariable(f"Missed_Quota_{node_id}_{prod.value}", lowBound=0)
                solver.add_to_objective(slack_quota * P_CONTRACT)
                solver.add_custom_constraint(
                    (prod_out + slack_quota) >= required_amount,
                    f"Processing_Quota_{node_id}_{prod.value}"
                )


# Port Constraint

In [13]:
for node_id, node in network.nodes.items():        
    if node.type == NodeType.PORT:
        for prod in ProductType:
            flow_in = pulp.lpSum(solver.get_inbound_flows(node_id, prod))
            flow_out = pulp.lpSum(solver.get_outbound_flows(node_id, prod))
            target_volume = node.contract_demands.get(prod, 0)

            exported = pulp.LpVariable(f"Exported_{node_id}_{prod.value}", lowBound=0, upBound=target_volume)
            export_vars[(node_id, prod)] = exported
            solver.add_to_objective(exported * EXPORT_REWARD)
            
            solver.add_custom_constraint((flow_in) == (exported), f"Port_Bal_{node_id}_{prod.value}")
            solver.add_custom_constraint(flow_out == 0, f"Port_No_Out_{node_id}_{prod.value}")

# Fixed flow flows

In [14]:
count = 0
for i, constr in enumerate(network.constraints):
    # Fetches relevant variables by filtering flow_vars
    relevant_vars = [
        var for (src, dst, mode, prod), var in solver.flow_vars.items()
        if src == constr.source_id and dst == constr.target_id and prod == constr.product
    ]
    
    var_name = f"{constr.source_id}_{constr.target_id}".replace("-", "_")
    slack_var = pulp.LpVariable(f"Missed_FixedFlow_{i}_{var_name}", lowBound=0)
    solver.add_to_objective(slack_var * P_CONTRACT)

    expr = pulp.lpSum(relevant_vars) + slack_var
    constr_name = f"FixedFlow_{i}_{var_name}_{constr.product.value}"

    try:
        if constr.type == 'min':
            solver.add_custom_constraint(expr >= constr.volume, constr_name)
        elif constr.type == 'equal':
            solver.add_custom_constraint(expr == constr.volume, constr_name)
        elif constr.type == 'max':
            solver.add_custom_constraint(expr <= constr.volume, constr_name)
        count += 1
    except Exception as e:
        print(e)
        continue

print(f"Applied {count} Fixed Flow rules.")

Applied 228 Fixed Flow rules.


# Run Solver

In [15]:
status = solver.solve(msg=1, threads=4)
print(f"Optimization finished with status: {pulp.LpStatus[status]}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/42/_z7t3f3n2b74bggwwctg77z80000gn/T/66c2083022ad482b8dccf55d4b6c47a8-pulp.mps -threads 4 -timeMode elapsed -branch -printingOptions all -solution /var/folders/42/_z7t3f3n2b74bggwwctg77z80000gn/T/66c2083022ad482b8dccf55d4b6c47a8-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 21926 COLUMNS
At line 96787689 RHS
At line 96809611 BOUNDS
At line 129736394 ENDATA
Problem MODEL has 21921 rows, 32931158 columns and 96761329 elements
Coin0008I MODEL read with 0 errors
threads was changed from 0 to 4
Option for timeMode changed from cpu to elapsed
Presolve 14316 (-7605) rows, 14789421 (-18141737) columns and 44035965 (-52725364) elements
Perturbing problem by 0.001% of 111480.39 - largest nonzero change 0.51412263 ( 0.00046117763%) - largest zero change 0.22315913
0  Obj 1.21127

# Saving Solution

In [16]:
import json

solution = {
    "status": pulp.LpStatus[status],
    "objective": pulp.value(solver.prob.objective),
    "variables": {v.name: v.varValue for v in solver.prob.variables()}
}

with open("data/2023/lp_solution.json", "w") as f: json.dump(solution, f, indent=4)

In [18]:
import polars as pl

TOLERANCE = 0.01

def extract_active_vars(var_dict, var_type):
    results = []
    
    for key, var in var_dict.items():
        # Keep your memory-saving pre-filter
        if var.varValue and var.varValue > TOLERANCE:
            
            # 1. Flow Variables: (src, dst, mode, product)
            if len(key) == 4:
                src, dst, mode, prod = key
                node_id = src
                destination = dst
                prod_val = prod.value
                
            # 2. Slack Variables: (node_id, slack_type, product)
            elif len(key) == 3:
                n_id, slack_type, prod = key
                node_id = n_id
                destination = None  # Slacks don't have a destination
                prod_val = prod.value
                
            # 3. Storage Variables: (node_id, product)
            elif len(key) == 2:
                n_id, prod = key
                node_id = n_id
                destination = None  # Storage doesn't have a destination
                prod_val = prod.value
                
            else:
                raise ValueError(f"Unexpected key length {len(key)} for key {key}")

            # Append the tuple matching your exact Polars schema:
            # ["type", "node_id", "destination", "product", "volume", "var_solver"]
            results.append((var_type, node_id, destination, prod_val, var.varValue, var.name))
            
    return results

# 2. Build the Polars DataFrame directly from the combined lists
df_results = pl.DataFrame(
    extract_active_vars(solver.flow_vars, "FLOW") + extract_active_vars(solver.slack_vars, "SLACK"),
    schema=["type", "node_id", "destination", "product", "volume", "var_solver"]
)

df_results.write_parquet('data/2023/lp_solution.parquet')

/var/folders/42/_z7t3f3n2b74bggwwctg77z80000gn/T/ipykernel_12533/4289367327.py:43: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df_results = pl.DataFrame(


# Load LP solution

In [19]:
df_solution = pl.read_parquet('data/2023/lp_solution.parquet')


In [27]:
df_solution.filter(pl.col('destination').str.starts_with('PORT'))[['volume']].to_pandas().sum()

volume   116,366,730
dtype: float64

In [37]:
(
    df_solution
    .filter(pl.col('destination').str.starts_with('PROCES'))
    .group_by('product')
    .agg(pl.col('volume').sum())
    .to_pandas()
)

,product,volume
0,SOYBEANS,"57,747,850"
1,SOYBEAN_CAKE,"22,609,623"
2,SOYBEAN_OIL,"8,284,419"


In [28]:
df_solution.filter(pl.col('node_id').str.starts_with('PROD'))[['volume']].to_pandas().sum()

volume   152,144,238
dtype: float64